In [ ]:
# %%
# SECTION 1: Imports
# -------------------
# Loads all required libraries:
#   - selenium webdriver: controls a real Chrome browser to load JS-rendered pages
#   - WebDriverWait + EC: waits for a specific element to appear before scraping
#   - BeautifulSoup: parses the rendered HTML to extract data
#   - csv: writes extracted records to a CSV file
#
# Requirements: pip install selenium beautifulsoup4
# Chrome + ChromeDriver must be installed and on PATH.
# Alternatively: pip install webdriver-manager (see Section 3 for setup)

import csv

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

print("Imports loaded successfully.")

In [ ]:
# %%
# SECTION 2: Configuration
# -------------------------
# Set your scrape target and options here before running any section below.
#
# url          - The page you want to scrape
# wait_for     - CSS selector that only appears once JS has fully rendered the data.
#                The browser will wait until this element exists before scraping.
#                Example: "span[databind='text: name']" waits for the first product span.
# timeout      - Max seconds to wait for wait_for to appear (increase for slow pages)
# headless     - True: no browser window (faster). False: shows browser (easier to debug).
# parent_tag   - HTML tag wrapping each record
# parent_class - CSS class to filter parent elements (None to match all)
# child_tag    - Tag to search within each parent element
# fields       - List of (column_name, attr_name, attr_value) tuples
# output_path  - Where to save the resulting CSV

url          = 'https://www.cdms.net/Label-Database/Advanced-Search#Result-products'
wait_for     = "span[databind='text: name']"  # CSS selector to confirm JS has loaded
timeout      = 15    # seconds
headless     = True  # Set to False to watch the browser while debugging

parent_tag   = 'span'
parent_class = None
child_tag    = 'span'
fields       = [
    ('brand_name',          'databind', 'text: name'),
    ('registration_number', 'databind', 'text: regNumber'),
    # ('column_name', 'attr_name', 'attr_value'),
]
output_path  = 'active_ingredients.csv'

print(f"Config ready. Target: {url}")
print(f"Waiting for: '{wait_for}' (timeout: {timeout}s)")

In [ ]:
# %%
# SECTION 3: build_driver()
# --------------------------
# Creates and returns a configured Chrome WebDriver instance.
# Sets headless mode and common flags to avoid common Linux/Docker issues.
# If ChromeDriver is not on your PATH, uncomment the webdriver-manager lines
# to have it downloaded and managed automatically.

def build_driver(headless: bool):
    options = Options()
    if headless:
        options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-gpu')
    options.add_argument('user-agent=Mozilla/5.0')

    # --- Uncomment below to auto-manage ChromeDriver ---
    # from webdriver_manager.chrome import ChromeDriverManager
    # from selenium.webdriver.chrome.service import Service
    # return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

    return webdriver.Chrome(options=options)

print("build_driver() defined.")

In [ ]:
# %%
# SECTION 4: fetch_page_selenium()
# ---------------------------------
# Launches Chrome, navigates to the configured URL, and waits for the 'wait_for'
# CSS selector to appear in the DOM. Once found, captures the full rendered HTML
# and closes the browser. If the selector never appears (timeout), it captures
# whatever HTML did load so you can inspect what went wrong.
# The result is stored in 'html' for use in the parse step.

def fetch_page_selenium(url, wait_for, timeout, headless):
    driver = build_driver(headless)
    try:
        driver.get(url)
        WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, wait_for))
        )
        print(f"Selector '{wait_for}' found — page fully loaded.")
        return driver.page_source
    except TimeoutException:
        print(f"Timeout: '{wait_for}' did not appear within {timeout}s.")
        print("Returning partial HTML. Try increasing timeout or changing wait_for.")
        return driver.page_source
    finally:
        driver.quit()

print(f"Fetching: {url}")
html = fetch_page_selenium(url, wait_for, timeout, headless)
print(f"HTML captured. ({len(html)} chars)")

In [ ]:
# %%
# SECTION 5: Parse rendered HTML
# -------------------------------
# Parses the rendered HTML string from fetch_page_selenium() into a BeautifulSoup
# object. Unlike the requests-based scraper, the HTML here includes content that
# was injected by JavaScript, so data fields should now be visible.
# Prints a preview — you should see actual product/record data if the page loaded correctly.

soup = BeautifulSoup(html, 'html.parser')
print("HTML parsed.")
print("Preview (first 500 chars of text):")
print(soup.get_text()[:500])

In [ ]:
# %%
# SECTION 6: Inspect parent elements
# ------------------------------------
# Finds parent elements and prints the first few raw so you can verify the structure
# matches what you expect. If results look wrong, adjust parent_tag / parent_class
# in Section 2 and re-run from Section 4. Useful for understanding the page layout
# before committing to a field definition.

kwargs = {'class_': parent_class} if parent_class else {}
parents = soup.find_all(parent_tag, **kwargs)

print(f"Found {len(parents)} '{parent_tag}' elements.")
print("\nFirst 3 (raw HTML):")
for p in parents[:3]:
    print(p)
    print()

In [ ]:
# %%
# SECTION 7: scrape_records()
# ----------------------------
# Iterates over every parent element and extracts child elements by matching
# attribute name/value pairs defined in 'fields'. Builds one dict per parent
# and collects them into 'records'. Skips parents where all fields are empty.
# Preview the first 5 records to confirm data looks right before writing the CSV.

def scrape_records(soup, parent_tag, parent_class, child_tag, fields):
    kwargs = {'class_': parent_class} if parent_class else {}
    parents = soup.find_all(parent_tag, **kwargs)
    records = []
    for parent in parents:
        record = {}
        for col_name, attr_name, attr_value in fields:
            child = parent.find(child_tag, attrs={attr_name: attr_value})
            record[col_name] = child.get_text(strip=True) if child else ''
        if any(record.values()):
            records.append(record)
    return records

records = scrape_records(soup, parent_tag, parent_class, child_tag, fields)
print(f"Extracted {len(records)} records.")
print("\nFirst 5:")
for r in records[:5]:
    print(r)

In [ ]:
# %%
# SECTION 8: write_csv()
# -----------------------
# Writes the extracted records to a multi-column CSV at output_path.
# Column headers are taken from the field names in Section 2.
# Run this last, after confirming the Section 7 preview looks correct.

def write_csv(records, fieldnames, output_path):
    with open(output_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(records)
    print(f"Wrote {len(records)} records to '{output_path}'")

if records:
    write_csv(records, [f[0] for f in fields], output_path)
else:
    print("No records to write. Check your field definitions or wait_for selector.")